In [5]:
import pandas as pd
import numpy as np

# =========================
# Your target architecture
# =========================
COMMON_FEATURES = [
    "flow_duration",
    "protocol",
    "service_state",
    "fwd_packet_count",
    "bwd_packet_count",
    "fwd_bytes",
    "bwd_bytes",
    "packet_rate",
    "byte_rate",
    "avg_packet_size",
    "syn_flag_count",
    "rst_flag_count",
    "ack_flag_count",
]

# =========================
# Load your NSL-KDD CSV
# =========================
# CHANGE PATH if needed
df = pd.read_csv("Datasets/NSL_KDD_test.csv")

# =========================
# Required NSL-KDD columns for this mapping
# =========================
required = ["duration","protocol_type","service","flag","src_bytes","dst_bytes","count","srv_count"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required NSL-KDD columns for mapping: {missing}")

# -------------------------
# Clean categoricals
# -------------------------
for c in ["protocol_type", "service", "flag"]:
    df[c] = (
        df[c]
        .astype("string")
        .str.strip()
        .replace({"": "none", "nan": "none", "None": "none", "-": "none"})
        .fillna("none")
    )

# -------------------------
# Coerce numeric inputs
# -------------------------
for c in ["duration","src_bytes","dst_bytes","count","srv_count"]:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

# -------------------------
# Map into COMMON_FEATURES
# -------------------------
flow_duration = df["duration"].astype(float)
protocol = df["protocol_type"].astype("string")

# Best available analog in NSL-KDD: service + flag
service_state = df["service"].astype("string") + "_" + df["flag"].astype("string")

# NSL-KDD does not have packet totals; use zeros (or you can choose another proxy)
fwd_packet_count = np.zeros(len(df), dtype=float)
bwd_packet_count = np.zeros(len(df), dtype=float)

fwd_bytes = df["src_bytes"].astype(float)
bwd_bytes = df["dst_bytes"].astype(float)
total_bytes = (fwd_bytes + bwd_bytes).to_numpy()

# byte_rate = bytes/sec
byte_rate = np.divide(
    total_bytes,
    flow_duration.to_numpy(),
    out=np.zeros(len(df), dtype=float),
    where=flow_duration.to_numpy() > 0
)

# packet_rate proxy: connections/sec to same host (count is not packets, but best available)
packet_rate = np.divide(
    df["count"].to_numpy(),
    flow_duration.to_numpy(),
    out=np.zeros(len(df), dtype=float),
    where=flow_duration.to_numpy() > 0
)

# avg_packet_size proxy: bytes per connection (count is connections)
avg_packet_size = np.divide(
    total_bytes,
    df["count"].to_numpy(),
    out=np.zeros(len(df), dtype=float),
    where=df["count"].to_numpy() > 0
)

# NSL-KDD doesn't include per-flag counts; set to 0
syn_flag_count = np.zeros(len(df), dtype=int)
rst_flag_count = np.zeros(len(df), dtype=int)
ack_flag_count = np.zeros(len(df), dtype=int)

df_common = pd.DataFrame({
    "flow_duration": flow_duration,
    "protocol": protocol,
    "service_state": service_state,
    "fwd_packet_count": fwd_packet_count,
    "bwd_packet_count": bwd_packet_count,
    "fwd_bytes": fwd_bytes,
    "bwd_bytes": bwd_bytes,
    "packet_rate": packet_rate,
    "byte_rate": byte_rate,
    "avg_packet_size": avg_packet_size,
    "syn_flag_count": syn_flag_count,
    "rst_flag_count": rst_flag_count,
    "ack_flag_count": ack_flag_count,
})[COMMON_FEATURES]

# Optional: keep class label if present
if "class" in df.columns:
    df_common["class"] = df["class"].astype("string")
# Optional: also create a binary label (0 normal, 1 anomaly) if your class uses those strings
if "class" in df.columns:
    s = df["class"].astype("string").str.strip().str.lower()
    df_common["label_bin"] = np.where(s.isin(["normal", "normal."]), 0, 1)

# Save
out_path = "Datasets/NSL_KDD_common_features_test.csv"
df_common.to_csv(out_path, index=False)

print("Done.")
print("Output:", out_path)
print("Shape:", df_common.shape)
df_common.head()

Done.
Output: Datasets/NSL_KDD_common_features_test.csv
Shape: (5039, 15)


,flow_duration,protocol,service_state,fwd_packet_count,bwd_packet_count,fwd_bytes,bwd_bytes,packet_rate,byte_rate,avg_packet_size,syn_flag_count,rst_flag_count,ack_flag_count,class,label_bin
0,0.0,icmp,ecr_i_SF,0.0,0.0,1032.0,0.0,0.0,0.0,79.384615,0,0,0,anomaly,1
1,0.0,tcp,http_REJ,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0,0,0,normal,0
2,0.0,tcp,private_S0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0,0,0,anomaly,1
3,0.0,udp,domain_u_SF,0.0,0.0,45.0,114.0,0.0,0.0,79.500000,0,0,0,normal,0
4,0.0,tcp,http_S0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0,0,0,anomaly,1
